In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive/')

import os
os.chdir('/content/drive/MyDrive/ml_v2/')

Mounted at /content/drive/


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from lightgbm import LGBMClassifier

# ========= 1. 讀資料 =========
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# ========= 1.5 反轉反向指標 =========
reverse_cols = [
    "llama_vagueness_score_1",
    "llama_deflection_score_1"
]

for col in reverse_cols:
    if col not in df.columns:
        raise ValueError(f"{col} 不存在於資料中，無法反轉")
    df[col] = 1 - df[col]

# ========= 2. target / groups =========
y = df["label"]
groups_all = df["Company"]

# ========= 3. feature groups =========
semantic_cols = [
    "llama_specificity_score_1",
    "llama_evidence_substantiation_score_1",
    "llama_vagueness_score_1",
    "llama_commitment_score_1",
    "llama_temporal_credibility_score_1",
    "llama_deflection_score_1",
    "llama_comparability_score_1"
]

lexical_cols = [
    "llama_has_scope_1",
    "llama_has_sbti_1",
    "llama_has_material_1",
    "llama_has_kpi_1",
    "llama_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# ========= 4. 檢查欄位 =========
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# ========= 5. Ablation sets =========
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# ========= 6. 類別不平衡 =========
pos = np.sum(y == 1)
neg = np.sum(y == 0)
scale_pos_weight = neg / pos if pos > 0 else 1

print(f"Positive class count: {pos}")
print(f"Negative class count: {neg}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# ========= 7. outer / inner CV (group-aware) =========
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# ========= 8. LightGBM pipeline =========
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LGBMClassifier(
        objective="binary",
        class_weight=None,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

# ========= 9. GridSearchCV 參數 =========
param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [-1, 3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [15, 31, 63],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8]
}

# ========= 10. 找最佳 threshold =========
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# ========= 11. 評估函數 =========
def evaluate_with_grouped_nested_cv_and_oof_threshold(X, y, groups, feature_name):
    fold_metrics = []
    best_params_list = []
    best_thresholds = []

    print(f"\n========== {feature_name} ==========")
    print(f"Num features: {X.shape[1]}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]

        # inner CV: 找最佳模型參數
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="f1",
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_train, y_train, **{"groups": groups_train})

        best_model = grid.best_estimator_
        best_params_list.append(grid.best_params_)

        # 用 outer train 內的 OOF probabilities 找 threshold
        oof_prob = cross_val_predict(
            estimator=best_model,
            X=X_train,
            y=y_train,
            groups=groups_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        best_threshold, best_f1 = find_best_threshold(y_train, oof_prob)
        best_thresholds.append(best_threshold)

        # 重新 fit outer training fold
        best_model.fit(X_train, y_train)

        # outer test 評估
        test_prob = best_model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"OOF_F1={best_f1:.4f} | "
            f"Test_F1={fold_result['f1']:.4f}"
        )
        print(f"[{feature_name}] Best params: {grid.best_params_}")

    # 各 fold 指標
    accs = [m["accuracy"] for m in fold_metrics]
    f1s = [m["f1"] for m in fold_metrics]
    rocs = [m["roc_auc"] for m in fold_metrics]
    precs = [m["precision"] for m in fold_metrics]
    recs = [m["recall"] for m in fold_metrics]
    pr_aucs = [m["average_precision"] for m in fold_metrics]

    return {
        "Model": "LightGBM",
        "Feature_Set": feature_name,
        "Num_Features": X.shape[1],

        "Accuracy_mean": np.mean(accs),
        "Accuracy_std": np.std(accs),

        "F1_mean": np.mean(f1s),
        "F1_std": np.std(f1s),

        "ROC_AUC_mean": np.mean(rocs),
        "ROC_AUC_std": np.std(rocs),

        "Precision_mean": np.mean(precs),
        "Precision_std": np.std(precs),

        "Recall_mean": np.mean(recs),
        "Recall_std": np.std(recs),

        "PR_AUC_mean": np.mean(pr_aucs),
        "PR_AUC_std": np.std(pr_aucs),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Threshold_std": np.std(best_thresholds),

        "Best_Params_Per_Fold": str(best_params_list)
    }

# ========= 12. 執行 =========
results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()
    results.append(
        evaluate_with_grouped_nested_cv_and_oof_threshold(
            X, y, groups_all, feature_name
        )
    )

results_df = pd.DataFrame(results)

# ========= 13. 四捨五入 =========
numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== Final Results =====")
print(results_df)

results_df.to_csv(
    "llama_LGBM_grouped_nestedCV_oof_threshold_reversed_with_std.csv",
    index=False,
    encoding="utf-8-sig"
)

Positive class count: 32
Negative class count: 296
scale_pos_weight: 9.2500

========== M1: Semantic ==========
Num features: 7
[M1: Semantic] Fold 1 started


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M1: Semantic] Fold 1 done | Threshold=0.45 | OOF_F1=0.5385 | Test_F1=0.5714
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 300, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M1: Semantic] Fold 2 started


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M1: Semantic] Fold 2 done | Threshold=0.82 | OOF_F1=0.5538 | Test_F1=0.5714
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M1: Semantic] Fold 3 started


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M6: Semantic + Lexical + Financial] Fold 8 done | Threshold=0.40 | OOF_F1=0.7812 | Test_F1=0.5000
[M6: Semantic + Lexical + Financial] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': -1, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M6: Semantic + Lexical + Financial] Fold 9 started


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M6: Semantic + Lexical + Financial] Fold 9 done | Threshold=0.33 | OOF_F1=0.7500 | Test_F1=0.6000
[M6: Semantic + Lexical + Financial] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 300, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M6: Semantic + Lexical + Financial] Fold 10 started
[M6: Semantic + Lexical + Financial] Fold 10 done | Threshold=0.41 | OOF_F1=0.7812 | Test_F1=0.2500
[M6: Semantic + Lexical + Financial] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': -1, 'model__n_estimators': 300, 'model__num_leaves': 31, 'model__subsample': 0.8}

===== Final Results =====
      Model                         Feature_Set  Num_Features  Accuracy_mean  \
0  LightGBM                        M1: Semantic             7         0.8647   
1  LightGBM                         M2: Lexical             5         0.8120   
2  LightGBM                       M3: Financial      

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score, average_precision_score
)
from lightgbm import LGBMClassifier

# ========= 1. 讀資料 =========
df = pd.read_csv("final_dataset_for_ml_FULL.csv", encoding="utf-8-sig")

# ========= 1.5 反轉反向指標 =========
reverse_cols = [
    "chatgpt_vagueness_score_1",
    "chatgpt_deflection_score_1"
]

for col in reverse_cols:
    if col not in df.columns:
        raise ValueError(f"{col} 不存在於資料中，無法反轉")
    df[col] = 1 - df[col]

# ========= 2. target / groups =========
y = df["label"]
groups_all = df["Company"]

# ========= 3. feature groups =========
semantic_cols = [
    "chatgpt_specificity_score_1",
    "chatgpt_evidence_substantiation_score_1",
    "chatgpt_vagueness_score_1",
    "chatgpt_commitment_score_1",
    "chatgpt_temporal_credibility_score_1",
    "chatgpt_deflection_score_1",
    "chatgpt_comparability_score_1"
]

lexical_cols = [
    "chatgpt_has_scope_1",
    "chatgpt_has_sbti_1",
    "chatgpt_has_material_1",
    "chatgpt_has_kpi_1",
    "chatgpt_has_percent_1"
]

financial_cols = [
    "SIZE",
    "ROA",
    "Leverage",
    "Cash_flow",
    "market_cap"
]

# ========= 4. 檢查欄位 =========
all_needed_cols = semantic_cols + lexical_cols + financial_cols + ["label", "Company"]
missing_cols = [col for col in all_needed_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"以下欄位不存在於資料中: {missing_cols}")

# ========= 5. Ablation sets =========
feature_sets = {
    "M1: Semantic": semantic_cols,
    "M2: Lexical": lexical_cols,
    "M3: Financial": financial_cols,
    "M4: Semantic + Lexical": semantic_cols + lexical_cols,
    "M5: Semantic + Financial": semantic_cols + financial_cols,
    "M6: Semantic + Lexical + Financial": semantic_cols + lexical_cols + financial_cols
}

# ========= 6. 類別不平衡 =========
pos = np.sum(y == 1)
neg = np.sum(y == 0)
scale_pos_weight = neg / pos if pos > 0 else 1

print(f"Positive class count: {pos}")
print(f"Negative class count: {neg}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

# ========= 7. outer / inner CV (group-aware) =========
outer_cv = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# ========= 8. LightGBM pipeline =========
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LGBMClassifier(
        objective="binary",
        class_weight=None,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

# ========= 9. GridSearchCV 參數 =========
param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [-1, 3, 5],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [15, 31, 63],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8]
}

# ========= 10. 找最佳 threshold =========
def find_best_threshold(y_true, y_prob):
    thresholds = np.arange(0.10, 0.91, 0.01)
    best_threshold = 0.50
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = t

    return best_threshold, best_score

# ========= 11. 評估函數 =========
def evaluate_with_grouped_nested_cv_and_oof_threshold(X, y, groups, feature_name):
    fold_metrics = []
    best_params_list = []
    best_thresholds = []

    print(f"\n========== {feature_name} ==========")
    print(f"Num features: {X.shape[1]}")

    for fold_idx, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups=groups), start=1
    ):
        print(f"[{feature_name}] Fold {fold_idx} started")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        groups_train = groups.iloc[train_idx]

        # inner CV: 找最佳模型參數
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=inner_cv,
            scoring="f1",
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_train, y_train, **{"groups": groups_train})

        best_model = grid.best_estimator_
        best_params_list.append(grid.best_params_)

        # 用 outer train 內的 OOF probabilities 找 threshold
        oof_prob = cross_val_predict(
            estimator=best_model,
            X=X_train,
            y=y_train,
            groups=groups_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]

        best_threshold, best_f1 = find_best_threshold(y_train, oof_prob)
        best_thresholds.append(best_threshold)

        # 重新 fit outer training fold
        best_model.fit(X_train, y_train)

        # outer test 評估
        test_prob = best_model.predict_proba(X_test)[:, 1]
        y_pred = (test_prob >= best_threshold).astype(int)

        fold_result = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, test_prob),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "average_precision": average_precision_score(y_test, test_prob)
        }
        fold_metrics.append(fold_result)

        print(
            f"[{feature_name}] Fold {fold_idx} done | "
            f"Threshold={best_threshold:.2f} | "
            f"OOF_F1={best_f1:.4f} | "
            f"Test_F1={fold_result['f1']:.4f}"
        )
        print(f"[{feature_name}] Best params: {grid.best_params_}")

    # 各 fold 指標
    accs = [m["accuracy"] for m in fold_metrics]
    f1s = [m["f1"] for m in fold_metrics]
    rocs = [m["roc_auc"] for m in fold_metrics]
    precs = [m["precision"] for m in fold_metrics]
    recs = [m["recall"] for m in fold_metrics]
    pr_aucs = [m["average_precision"] for m in fold_metrics]

    return {
        "Model": "LightGBM",
        "Feature_Set": feature_name,
        "Num_Features": X.shape[1],

        "Accuracy_mean": np.mean(accs),
        "Accuracy_std": np.std(accs),

        "F1_mean": np.mean(f1s),
        "F1_std": np.std(f1s),

        "ROC_AUC_mean": np.mean(rocs),
        "ROC_AUC_std": np.std(rocs),

        "Precision_mean": np.mean(precs),
        "Precision_std": np.std(precs),

        "Recall_mean": np.mean(recs),
        "Recall_std": np.std(recs),

        "PR_AUC_mean": np.mean(pr_aucs),
        "PR_AUC_std": np.std(pr_aucs),

        "Mean_Best_Threshold": np.mean(best_thresholds),
        "Threshold_std": np.std(best_thresholds),

        "Best_Params_Per_Fold": str(best_params_list)
    }

# ========= 12. 執行 =========
results = []

for feature_name, cols in feature_sets.items():
    X = df[cols].copy()
    results.append(
        evaluate_with_grouped_nested_cv_and_oof_threshold(
            X, y, groups_all, feature_name
        )
    )

results_df = pd.DataFrame(results)

# ========= 13. 四捨五入 =========
numeric_cols = results_df.select_dtypes(include=[np.number]).columns
results_df[numeric_cols] = results_df[numeric_cols].round(4)

print("\n===== Final Results =====")
print(results_df)

results_df.to_csv(
    "chatgpt_LGBM_grouped_nestedCV_oof_threshold_reversed_with_std.csv",
    index=False,
    encoding="utf-8-sig"
)

Positive class count: 32
Negative class count: 296
scale_pos_weight: 9.2500

========== M1: Semantic ==========
Num features: 7
[M1: Semantic] Fold 1 started


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M1: Semantic] Fold 1 done | Threshold=0.48 | OOF_F1=0.7273 | Test_F1=0.8889
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M1: Semantic] Fold 2 started


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M1: Semantic] Fold 2 done | Threshold=0.90 | OOF_F1=0.8000 | Test_F1=0.0000
[M1: Semantic] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M1: Semantic] Fold 3 started


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M6: Semantic + Lexical + Financial] Fold 8 done | Threshold=0.49 | OOF_F1=0.8966 | Test_F1=0.6667
[M6: Semantic + Lexical + Financial] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M6: Semantic + Lexical + Financial] Fold 9 started


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[M6: Semantic + Lexical + Financial] Fold 9 done | Threshold=0.75 | OOF_F1=0.8421 | Test_F1=1.0000
[M6: Semantic + Lexical + Financial] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__num_leaves': 15, 'model__subsample': 0.8}
[M6: Semantic + Lexical + Financial] Fold 10 started
[M6: Semantic + Lexical + Financial] Fold 10 done | Threshold=0.10 | OOF_F1=0.8621 | Test_F1=0.4444
[M6: Semantic + Lexical + Financial] Best params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.05, 'model__max_depth': -1, 'model__n_estimators': 300, 'model__num_leaves': 15, 'model__subsample': 0.8}

===== Final Results =====
      Model                         Feature_Set  Num_Features  Accuracy_mean  \
0  LightGBM                        M1: Semantic             7         0.9292   
1  LightGBM                         M2: Lexical             5         0.8120   
2  LightGBM                       M3: Financial     

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
